
# CIFAR-10 VAE + Local PCA Tangent-Space Attack

This notebook is the CIFAR-10 version of the MNIST pipeline.

Main pipeline:

\[
x \rightarrow E(x)=z
\rightarrow \widehat T_z\mathcal M_y
\rightarrow \delta_T = Ba
\rightarrow z' = z+\delta_T
\rightarrow D(z')
\rightarrow \text{victim classifier}
\]

Default targeted example:

- source = **cat** (`3`)
- target = **dog** (`5`)

The code keeps the same important variable names used in the MNIST notebook:
`model`, `victim`, `train_z`, `train_y`, `B`, `a`, `z0`, `x_recon`.


In [ ]:

# ============================================================
# CELL 1 — IMPORTS + CIFAR-10 DATA
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.utils import save_image

import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CIFAR_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Plain data for VAE, latent extraction, testing
plain_transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=plain_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=plain_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Separate augmented loader for victim training
victim_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

victim_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=victim_train_transform
)

victim_train_loader = DataLoader(
    victim_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Train:", len(train_dataset))
print("Test :", len(test_dataset))
print(CIFAR_CLASSES)


In [ ]:

# ============================================================
# CELL 2 — CIFAR-10 CONVOLUTIONAL VAE
# ============================================================

LATENT_DIM = 64

class CIFARConvVAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()

        # 32 -> 16 -> 8 -> 4
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU(),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)

        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)

        # 4 -> 8 -> 16 -> 32
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.reshape(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = self.decoder_fc(z)
        h = h.reshape(-1, 256, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z

model = CIFARConvVAE(LATENT_DIM).to(device)
print(model)


In [ ]:

# ============================================================
# CELL 3 — TRAIN VAE
# ============================================================

# CIFAR is much harder to reconstruct than MNIST.
# A smaller beta helps prioritize reconstruction quality.
BETA = 0.05

def vae_loss(x_hat, x, mu, logvar):
    recon_loss = F.mse_loss(
        x_hat,
        x,
        reduction="sum"
    )

    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    total = recon_loss + BETA * kl_loss
    return total, recon_loss, kl_loss


optimizer_vae = torch.optim.Adam(
    model.parameters(),
    lr=2e-4
)

VAE_EPOCHS = 30   # reduce to 15 for a quicker first test

for epoch in range(VAE_EPOCHS):
    model.train()

    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    for x, _ in train_loader:
        x = x.to(device)

        optimizer_vae.zero_grad()

        x_hat, mu, logvar, z = model(x)
        loss, recon, kl = vae_loss(x_hat, x, mu, logvar)

        loss.backward()
        optimizer_vae.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    n = len(train_dataset)

    print(
        f"Epoch {epoch+1:02d}/{VAE_EPOCHS} | "
        f"loss/img={total_loss/n:.4f} | "
        f"recon/img={total_recon/n:.4f} | "
        f"KL/img={total_kl/n:.4f}"
    )


In [ ]:

# ============================================================
# CELL 4 — VISUALIZE CIFAR RECONSTRUCTIONS
# ============================================================

model.eval()

x, y = next(iter(test_loader))
x = x[:10].to(device)

with torch.no_grad():
    mu, _ = model.encode(x)
    reconstructed = model.decode(mu)

fig, axes = plt.subplots(2, 10, figsize=(16, 4))

for i in range(10):
    axes[0, i].imshow(x[i].permute(1, 2, 0).cpu())
    axes[0, i].axis("off")

    axes[1, i].imshow(reconstructed[i].permute(1, 2, 0).cpu().clamp(0,1))
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Recon")
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# CELL 5 — EXTRACT DETERMINISTIC LATENT MEANS
# ============================================================

def extract_latents(model, loader):
    model.eval()

    latents = []
    labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            mu, _ = model.encode(x)

            latents.append(mu.cpu())
            labels.append(y.cpu())

    return (
        torch.cat(latents, dim=0),
        torch.cat(labels, dim=0)
    )

# Important: training latents come from NON-AUGMENTED images.
train_z, train_y = extract_latents(model, train_loader)
test_z, test_y = extract_latents(model, test_loader)

print("train_z:", train_z.shape)
print("train_y:", train_y.shape)
print("test_z :", test_z.shape)

# Optional latent 5-NN diagnostic
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_z.numpy(), train_y.numpy())

pred = knn.predict(test_z.numpy())
knn_acc = accuracy_score(test_y.numpy(), pred)

print(f"5-NN latent accuracy: {100*knn_acc:.2f}%")


In [ ]:

# ============================================================
# CELL 6 — CIFAR-10 VICTIM MODEL (RESNET-18)
# ============================================================

class CIFARVictim(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)

        # CIFAR-specific stem
        backbone.conv1 = nn.Conv2d(
            3, 64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        backbone.maxpool = nn.Identity()
        backbone.fc = nn.Linear(backbone.fc.in_features, 10)

        self.backbone = backbone

        # Internal normalization lets the rest of the notebook
        # keep all images in [0,1].
        self.register_buffer(
            "mean",
            torch.tensor([0.4914, 0.4822, 0.4465]).view(1,3,1,1)
        )
        self.register_buffer(
            "std",
            torch.tensor([0.2470, 0.2435, 0.2616]).view(1,3,1,1)
        )

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.backbone(x)

victim = CIFARVictim().to(device)

optimizer_cls = torch.optim.AdamW(
    victim.parameters(),
    lr=1e-3,
    weight_decay=5e-4
)

CLS_EPOCHS = 20

for epoch in range(CLS_EPOCHS):
    victim.train()

    correct = 0
    total = 0
    running_loss = 0.0

    for x, y in victim_train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer_cls.zero_grad()

        logits = victim(x)
        loss = F.cross_entropy(logits, y)

        loss.backward()
        optimizer_cls.step()

        running_loss += loss.item() * x.size(0)

        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    print(
        f"Epoch {epoch+1:02d}/{CLS_EPOCHS} | "
        f"loss={running_loss/total:.4f} | "
        f"train acc={100*correct/total:.2f}%"
    )

# Test accuracy
victim.eval()
correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        pred = victim(x).argmax(dim=1)

        correct += (pred == y).sum().item()
        total += y.size(0)

print(f"Victim test accuracy: {100*correct/total:.2f}%")


In [ ]:

# ============================================================
# CELL 7 — COMPLETE CIFAR TANGENT-SPACE LATENT ATTACK
# ============================================================

# Default: cat -> dog
SOURCE_CLASS = 3
TARGET_CLASS = 5

K_NEIGHBORS = 300
TANGENT_VARIANCE_THRESHOLD = 0.80
MAX_TANGENT_DIM = 24

ATTACK_STEPS = 1000
LEARNING_RATE = 0.02

KAPPA = 0.05
LAMBDA_NORM = 0.10
BETA_SEM = 15.0

ALPHA_STEPS = 201
SKIP_VALID_SAMPLES = 0

print("=" * 72)
print("CIFAR-10 TANGENT-SPACE LATENT ATTACK")
print("=" * 72)
print(
    f"Source = {SOURCE_CLASS} ({CIFAR_CLASSES[SOURCE_CLASS]}) | "
    f"Target = {TARGET_CLASS} ({CIFAR_CLASSES[TARGET_CLASS]})"
)

model.eval()
victim.eval()

for p in model.parameters():
    p.requires_grad = False

for p in victim.parameters():
    p.requires_grad = False


# ------------------------------------------------------------
# STEP 1 — find a clean source image for which BOTH
# original and VAE reconstruction are classified correctly.
# ------------------------------------------------------------

x_original = None
x_recon = None
z0 = None

valid_seen = 0

with torch.no_grad():

    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        for i in range(xb.size(0)):

            if yb[i].item() != SOURCE_CLASS:
                continue

            x_candidate = xb[i:i+1]

            pred_original = victim(
                x_candidate
            ).argmax(1).item()

            if pred_original != SOURCE_CLASS:
                continue

            mu, _ = model.encode(x_candidate)
            recon_candidate = model.decode(mu)

            pred_recon = victim(
                recon_candidate
            ).argmax(1).item()

            if pred_recon != SOURCE_CLASS:
                continue

            valid_seen += 1

            if valid_seen <= SKIP_VALID_SAMPLES:
                continue

            x_original = x_candidate.detach()
            x_recon = recon_candidate.detach()
            z0 = mu.detach()
            break

        if x_original is not None:
            break


if x_original is None:
    raise RuntimeError(
        "No suitable sample found. "
        "First inspect VAE reconstruction quality. "
        "You can also change SOURCE_CLASS."
    )


print("\nSTEP 1")
print("Victim(original) =", victim(x_original).argmax(1).item())
print("Victim(recon)    =", victim(x_recon).argmax(1).item())
print("z0 shape         =", tuple(z0.shape))


# ------------------------------------------------------------
# STEP 2 — nearest SAME-CLASS latent points
# ------------------------------------------------------------

Z_train = train_z.detach().cpu().float()
Y_train = train_y.detach().cpu().long().view(-1)

Z_same = Z_train[Y_train == SOURCE_CLASS]
z0_cpu = z0.detach().cpu().float()

distances = torch.norm(
    Z_same - z0_cpu,
    dim=1
)

k_local = min(K_NEIGHBORS, len(Z_same))

nearest_indices = torch.topk(
    distances,
    k=k_local,
    largest=False
).indices

Z_local = Z_same[nearest_indices]

print("\nSTEP 2")
print("Same-class points =", len(Z_same))
print("Local neighbors   =", tuple(Z_local.shape))


# ------------------------------------------------------------
# STEP 3 — LOCAL PCA / SVD
# estimate tangent basis B
# ------------------------------------------------------------

local_mean = Z_local.mean(
    dim=0,
    keepdim=True
)

X_local = Z_local - local_mean

U, S, Vh = torch.linalg.svd(
    X_local,
    full_matrices=False
)

variance = S ** 2
explained_variance = variance / variance.sum()
cumulative_variance = torch.cumsum(
    explained_variance,
    dim=0
)

idx = torch.where(
    cumulative_variance >= TANGENT_VARIANCE_THRESHOLD
)[0]

if len(idx):
    tangent_dim = idx[0].item() + 1
else:
    tangent_dim = Z_train.shape[1]

tangent_dim = max(2, tangent_dim)
tangent_dim = min(
    tangent_dim,
    MAX_TANGENT_DIM,
    Z_train.shape[1]
)

B = (
    Vh[:tangent_dim]
    .T
    .to(device)
    .float()
)

print("\nSTEP 3")
print("Tangent dimension =", tangent_dim)
print(
    "Explained variance =",
    f"{100*cumulative_variance[tangent_dim-1].item():.2f}%"
)
print("B shape =", tuple(B.shape))
print(
    "||B^T B - I|| =",
    torch.norm(
        B.T @ B -
        torch.eye(tangent_dim, device=device)
    ).item()
)


# ------------------------------------------------------------
# STEP 4 — typical latent scale
# ------------------------------------------------------------

train_center = Z_train.mean(dim=0, keepdim=True)

latent_scale = torch.norm(
    Z_train - train_center,
    dim=1
).median().item()

print("\nTypical latent scale =", latent_scale)


# ------------------------------------------------------------
# STEP 5 — optimize a
#
# delta_T = a B^T
# ------------------------------------------------------------

a = torch.zeros(
    (1, tangent_dim),
    device=device,
    requires_grad=True
)

optimizer = torch.optim.Adam(
    [a],
    lr=LEARNING_RATE
)

history = {
    "loss": [],
    "margin": [],
    "norm": [],
    "mse": [],
    "pred": []
}

best_success = None
best_success_norm = float("inf")

for step in range(ATTACK_STEPS):

    optimizer.zero_grad()

    delta_tangent = a @ B.T
    z_adv = z0 + delta_tangent
    x_adv = model.decode(z_adv)

    logits = victim(x_adv)

    target_logit = logits[:, TARGET_CLASS]

    logits_other = logits.clone()
    logits_other[:, TARGET_CLASS] = -1e9
    max_other = logits_other.max(dim=1).values

    # target should beat all other classes
    margin_loss = F.relu(
        max_other - target_logit + KAPPA
    ).mean()

    latent_norm = torch.norm(
        delta_tangent,
        p=2
    )

    normalized_norm = latent_norm / (
        latent_scale + 1e-8
    )

    # Only a proxy for perceptual/semantic preservation.
    image_mse = F.mse_loss(
        x_adv,
        x_recon
    )

    loss = (
        margin_loss
        + LAMBDA_NORM * normalized_norm
        + BETA_SEM * image_mse
    )

    loss.backward()
    optimizer.step()

    with torch.no_grad():
        delta_eval = a @ B.T
        x_eval = model.decode(z0 + delta_eval)

        pred = victim(
            x_eval
        ).argmax(1).item()

        norm_now = torch.norm(
            delta_eval,
            p=2
        ).item()

        if (
            pred == TARGET_CLASS
            and norm_now < best_success_norm
        ):
            best_success_norm = norm_now

            best_success = {
                "step": step,
                "a": a.detach().clone(),
                "delta": delta_eval.detach().clone(),
                "image": x_eval.detach().clone(),
                "pred": pred,
                "norm": norm_now
            }

    history["loss"].append(loss.item())
    history["margin"].append(margin_loss.item())
    history["norm"].append(norm_now)
    history["mse"].append(image_mse.item())
    history["pred"].append(pred)

    if step % 100 == 0:
        print(
            f"step={step:4d} | "
            f"pred={CIFAR_CLASSES[pred]:10s} | "
            f"margin={margin_loss.item():.4f} | "
            f"||delta||={norm_now:.4f} | "
            f"MSE={image_mse.item():.6f}"
        )


# ------------------------------------------------------------
# STEP 6 — choose final successful point if available
# ------------------------------------------------------------

if best_success is not None:
    a_final = best_success["a"]
    delta_final = best_success["delta"]
    x_final = best_success["image"]
    pred_final = best_success["pred"]

    print("\nTARGETED ATTACK SUCCESS")
    print("Best step =", best_success["step"])
else:
    with torch.no_grad():
        a_final = a.detach().clone()
        delta_final = a_final @ B.T
        x_final = model.decode(z0 + delta_final)
        pred_final = victim(x_final).argmax(1).item()

    print("\nNo targeted success found with current hyperparameters.")


# ------------------------------------------------------------
# STEP 7 — tangent-membership numerical check
# ------------------------------------------------------------

delta_projected = (
    delta_final @ B @ B.T
)

normal_residual = (
    delta_final - delta_projected
)

print("\nFINAL")
print(
    "Victim prediction =",
    pred_final,
    CIFAR_CLASSES[pred_final]
)
print(
    "||delta||_2 =",
    torch.norm(delta_final, p=2).item()
)
print(
    "Normal residual ||delta - Proj_T(delta)|| =",
    torch.norm(normal_residual, p=2).item()
)


# ------------------------------------------------------------
# STEP 8 — visualization
# ------------------------------------------------------------

difference = (
    x_final - x_recon
).abs()

fig, axes = plt.subplots(
    1, 4,
    figsize=(14, 4)
)

axes[0].imshow(
    x_original[0]
    .permute(1,2,0)
    .cpu()
    .clamp(0,1)
)
axes[0].set_title(
    f"Original\n{CIFAR_CLASSES[SOURCE_CLASS]}"
)

axes[1].imshow(
    x_recon[0]
    .permute(1,2,0)
    .cpu()
    .clamp(0,1)
)
axes[1].set_title("VAE reconstruction")

axes[2].imshow(
    x_final[0]
    .permute(1,2,0)
    .cpu()
    .clamp(0,1)
)
axes[2].set_title(
    f"Tangent attack\nvictim={CIFAR_CLASSES[pred_final]}"
)

# Show channel-averaged absolute difference
axes[3].imshow(
    difference[0]
    .mean(dim=0)
    .cpu(),
    cmap="hot"
)
axes[3].set_title("|Attacked - Recon|")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# CELL 8 — FIND EARLIEST TARGET CROSSING ALONG FINAL DIRECTION
# ============================================================

alphas = np.linspace(0.0, 1.0, ALPHA_STEPS)

records = []
first_success = None

with torch.no_grad():

    for alpha in alphas:

        z_alpha = (
            z0 + float(alpha) * delta_final
        )

        x_alpha = model.decode(z_alpha)
        logits = victim(x_alpha)
        probs = torch.softmax(logits, dim=1)

        pred = logits.argmax(1).item()

        rmse = torch.sqrt(
            F.mse_loss(
                x_alpha,
                x_recon
            )
        ).item()

        record = {
            "alpha": float(alpha),
            "image": x_alpha.cpu(),
            "pred": pred,
            "target_prob": probs[0, TARGET_CLASS].item(),
            "rmse": rmse
        }

        records.append(record)

        if (
            pred == TARGET_CLASS
            and first_success is None
        ):
            first_success = record


if first_success is None:
    print("No target crossing found along this direction.")
else:
    print(
        "First target crossing:",
        f"alpha={first_success['alpha']:.4f}",
        "prediction=",
        CIFAR_CLASSES[first_success["pred"]],
        "RMSE=",
        first_success["rmse"]
    )

    x_first = first_success["image"]

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))

    axes[0].imshow(
        x_original[0].permute(1,2,0).cpu().clamp(0,1)
    )
    axes[0].set_title("Original")

    axes[1].imshow(
        x_recon[0].permute(1,2,0).cpu().clamp(0,1)
    )
    axes[1].set_title("Recon")

    axes[2].imshow(
        x_first[0].permute(1,2,0).clamp(0,1)
    )
    axes[2].set_title(
        f"First target flip\nalpha={first_success['alpha']:.3f}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================================
# CELL 9 — COMPARE L0-LIKE SPARSITY ON a VS delta = B a
# ============================================================

tangent_dim = B.shape[1]

STEPS = 800
LR = 0.02

LAMBDA_SPARSE = 0.10
SIGMA = 0.05

def smooth_l0(x, sigma=0.05):
    return torch.sum(
        1.0 - torch.exp(
            -(x ** 2) / (2.0 * sigma ** 2)
        )
    )

def run_sparse_tangent_attack(sparse_on):

    a_sparse = torch.zeros(
        (1, tangent_dim),
        device=device,
        requires_grad=True
    )

    opt = torch.optim.Adam(
        [a_sparse],
        lr=LR
    )

    best = None
    best_norm = float("inf")

    for step in range(STEPS):

        opt.zero_grad()

        delta = a_sparse @ B.T
        x_adv = model.decode(z0 + delta)

        logits = victim(x_adv)

        target_logit = logits[:, TARGET_CLASS]

        other = logits.clone()
        other[:, TARGET_CLASS] = -1e9
        max_other = other.max(dim=1).values

        margin_loss = F.relu(
            max_other - target_logit + KAPPA
        ).mean()

        norm_loss = (
            torch.norm(delta, p=2)
            / (latent_scale + 1e-8)
        )

        sem_loss = F.mse_loss(
            x_adv,
            x_recon
        )

        if sparse_on == "a":
            sparse_loss = smooth_l0(
                a_sparse,
                SIGMA
            )

        elif sparse_on == "delta":
            sparse_loss = smooth_l0(
                delta,
                SIGMA
            )

        else:
            raise ValueError(
                "sparse_on must be 'a' or 'delta'"
            )

        loss = (
            margin_loss
            + LAMBDA_SPARSE * sparse_loss
            + LAMBDA_NORM * norm_loss
            + BETA_SEM * sem_loss
        )

        loss.backward()
        opt.step()

        with torch.no_grad():
            delta_eval = a_sparse @ B.T
            x_eval = model.decode(
                z0 + delta_eval
            )

            pred = victim(
                x_eval
            ).argmax(1).item()

            norm_now = torch.norm(
                delta_eval,
                p=2
            ).item()

            if (
                pred == TARGET_CLASS
                and norm_now < best_norm
            ):
                best_norm = norm_now
                best = {
                    "a": a_sparse.detach().clone(),
                    "delta": delta_eval.detach().clone(),
                    "image": x_eval.detach().clone(),
                    "pred": pred
                }

    if best is None:
        with torch.no_grad():
            delta_eval = a_sparse @ B.T
            x_eval = model.decode(z0 + delta_eval)
            pred = victim(x_eval).argmax(1).item()

            best = {
                "a": a_sparse.detach().clone(),
                "delta": delta_eval.detach().clone(),
                "image": x_eval.detach().clone(),
                "pred": pred
            }

    return best


result_a = run_sparse_tangent_attack("a")
result_delta = run_sparse_tangent_attack("delta")

EPS = 1e-3

def l0_count(x):
    return (
        torch.abs(x) > EPS
    ).sum().item()

print("=" * 70)
print("L0-LIKE COMPARISON")
print("=" * 70)

print("\nA) SPARSITY ON a")
print(
    "prediction:",
    CIFAR_CLASSES[result_a["pred"]]
)
print("||a||_0 approx     =", l0_count(result_a["a"]))
print("||delta||_0 approx =", l0_count(result_a["delta"]))

print("\nB) SPARSITY ON delta")
print(
    "prediction:",
    CIFAR_CLASSES[result_delta["pred"]]
)
print("||a||_0 approx     =", l0_count(result_delta["a"]))
print("||delta||_0 approx =", l0_count(result_delta["delta"]))


fig, axes = plt.subplots(1, 3, figsize=(11, 3))

axes[0].imshow(
    x_recon[0].permute(1,2,0).cpu().clamp(0,1)
)
axes[0].set_title("Clean recon")

axes[1].imshow(
    result_a["image"][0]
    .permute(1,2,0)
    .cpu()
    .clamp(0,1)
)
axes[1].set_title(
    f"L0-like on a\nvictim={CIFAR_CLASSES[result_a['pred']]}"
)

axes[2].imshow(
    result_delta["image"][0]
    .permute(1,2,0)
    .cpu()
    .clamp(0,1)
)
axes[2].set_title(
    f"L0-like on delta\nvictim={CIFAR_CLASSES[result_delta['pred']]}"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# CELL 10 — SAVE ATTACK IMAGE FOR EXTERNAL VICTIM TESTING
# ============================================================

# Choose which image you want:
# attack_to_save = x_final
# attack_to_save = result_delta["image"]
attack_to_save = x_final

img = attack_to_save.detach().cpu().clamp(0, 1)

save_image(
    img,
    "cifar_tangent_attack_32x32.png"
)

# Large nearest-neighbor version:
# preserves the original CIFAR pixels without smoothing.
img_large = F.interpolate(
    img,
    size=(320, 320),
    mode="nearest"
)

save_image(
    img_large,
    "cifar_tangent_attack_320x320.png"
)

print("Saved:")
print("  cifar_tangent_attack_32x32.png")
print("  cifar_tangent_attack_320x320.png")

# Google Colab:
try:
    from google.colab import files
    files.download("cifar_tangent_attack_320x320.png")
except Exception:
    print("Not running in Colab; files are saved locally.")
